In [25]:
import re
import joblib
import numpy as np
import pandas as pd
import nltk
import ftfy
import os

import os
from dotenv import load_dotenv
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI


from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from scipy.sparse import hstack, csr_matrix


In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\rivin\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

# Emotion Lexicons

In [3]:
# These are predefined word sets which will be used to identify the specific emotional states in text
JOY_WORDS = {
    'happy','happiness','joy','joyful','excited','excitement','amazing','wonderful',
    'fantastic','great','love','loved','grateful','gratitude','blessed','thrilled',
    'delighted','ecstatic','overjoyed','proud','celebrate','celebrating','smiling',
    'laughing','laugh','beautiful','incredible','awesome','lucky','thankful',
    'promoted','promotion','scholarship','accepted','engaged','proposal','born',
    'baby','married','wedding','vacation','holiday','trip','surprise','gift',
    'moon','unstoppable','alive','peace','peaceful','magical','crossed','finish',
    'line','milestone','achievement','achieved','dream','worth','penny','saved',
    'cheering','dancing','danced','winning','won','passed','congratulations',
    'glowing','beaming','radiant','light','smile','grinning','grin','elated',
    'content','fulfilling','fulfilled','meaningful','rewarding','rewarded'
}

ANXIETY_WORDS = {
    'anxious','anxiety','worry','worried','worrying','panic','panicking','fear',
    'fearful','terrified','terrifying','terror','dread','dreading','nervous',
    'catastrophize','catastrophizing','overthink','overthinking','racing',
    'trembling','shaking','freeze','frozen','avoid','avoidance','scared',
    'breathe','chest','tightens','tighten','attacks','attack','scenarios',
    'convinced','wrong','shake','judged','highways','losing','control',
    'smaller','triggered','trigger','worrying','terrible','cancelled',
    'interviews','crowd','crowded','spiral','spiraling','restless','uneasy',
    'apprehensive','tense','tension','phobia','obsess','obsessing',
    'checking','locks','ruminate','ruminating','sleepless','insomnia',
    'heart','pounding','sweating','sweat','nauseous','nausea','faint'
}

STRESS_WORDS = {
    'stress','stressed','stressful','burnout','burned','overwhelmed','overloaded',
    'exhausted','exhaustion','pressure','deadline','workload','furious','angry',
    'anger','irritable','irritated','snapping','frustrated','frustration',
    'relentless','piling','humiliated','juggling','rent','cover','barely',
    'syllabus','missed','launch','client','sixteen','edge','temper','aches',
    'breaking','crushing','meetings','running','empty','behind','overdue',
    'manager','boss','colleague','fired','layoff','debt','bills','financial',
    'screaming','yelling','yelled','slammed','snapped','boiling','fuming',
    'seething','livid','outraged','grinding','drained','depleted','stretched',
    'pulled','pushed','collapsing','crumbling','falling','apart','cope','coping'
}

DEPRESSION_WORDS = {
    'depressed','depression','hopeless','hopelessness','empty','numb','worthless',
    'useless','lonely','loneliness','isolated','isolation','crying','cried',
    'sadness','sad','miserable','darkness','dark','pointless','meaningless',
    'void','hollow','broken','shattered','defeated','lost','invisible',
    'disconnected','ghost','drifting','faking','fake','mask','pretend',
    'exhausted','heavy','sinking','drowning','suffocating','trapped','stuck',
    'failure','failed','disappoint','disappointing','ashamed','shame','guilt',
    'regret','nothingness','bleak','grey','gray','colorless','lifeless',
    'apathy','apathetic','withdrawn','detached','unmotivated','unmoved'
}

SUICIDAL_WORDS = {
    'suicidal','suicide','die','dying','dead','death','kill','goodbye','note',
    'plan','end','bridge','pills','overdose','cut','cutting','harm','pain',
    'method','ready','disappear','disappeared','letgo','peace','relieved',
    'farewell','arranged','neighbor','dog','stopping','almost',
    'tonight','night','stockpiling','researched','methods','reason','stay',
    'birthday','unbearable','terminal','decided','terms','deeper','wakeup',
    'morning','hoped','awake','knife','rope','jump','jumped','ledge','final'
}

NEGATION_WORDS = {
    'not','never','no','nothing','nobody','nowhere','neither','nor','none',
    'cannot','cant','wont','wouldnt','shouldnt','couldnt','dont','doesnt',
    'didnt','hasnt','havent','hadnt','isnt','arent','wasnt','werent'
}


# Text cleaning functions

In [4]:
lemmatizer = WordNetLemmatizer()

def fix_encoding(text):
    if not isinstance(text, str):
        return text
    text = ftfy.fix_text(text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def remove_patterns(text):
    text = str(text).lower()
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_map = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'R': wordnet.ADV}
    return tag_map.get(tag, wordnet.NOUN)

def lemmatize_text(text):
    tokens = word_tokenize(text)
    result = []
    for token in tokens:
        if not token.isalpha():
            continue
        pos   = get_wordnet_pos(token)
        lemma = lemmatizer.lemmatize(token, pos)
        result.append(lemma)
    return ' '.join(result)

# Feature Engineering

In [5]:
def count_lexicon(text, lexicon):
    tokens = set(text.lower().split())
    return len(tokens & lexicon)

def has_negation(text):
    tokens = set(text.lower().split())
    return int(bool(tokens & NEGATION_WORDS))

def exclamation_count(text):
    return text.count('!')

def question_count(text):
    return text.count('?')

def caps_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for c in letters if c.isupper()) / len(letters)

def avg_word_length(text):
    words = text.split()
    if not words:
        return 0.0
    return sum(len(w) for w in words) / len(words)

def build_features(df):
    features = pd.DataFrame()
    features['num_chars']  = df['text'].str.len()
    features['num_sentences'] = df['text'].apply(lambda x: len(sent_tokenize(str(x))))
    features['num_words'] = df['text'].str.split().str.len()
    features['avg_word_len'] = df['text'].apply(avg_word_length)
    features['exclamation'] = df['text'].apply(exclamation_count)
    features['question']  = df['text'].apply(question_count)
    features['caps_ratio']  = df['text'].apply(caps_ratio)
    features['has_negation'] = df['text'].apply(has_negation)
    features['joy_score']  = df['text'].apply(lambda x: count_lexicon(x, JOY_WORDS))
    features['anxiety_score'] = df['text'].apply(lambda x: count_lexicon(x, ANXIETY_WORDS))
    features['stress_score'] = df['text'].apply(lambda x: count_lexicon(x, STRESS_WORDS))
    features['depression_score'] = df['text'].apply(lambda x: count_lexicon(x, DEPRESSION_WORDS))
    features['suicidal_score'] = df['text'].apply(lambda x: count_lexicon(x, SUICIDAL_WORDS))
    return features.values

# load Models

In [6]:
MODEL_DIR = '../models/'

lr_model = joblib.load(os.path.join(MODEL_DIR, 'lr_model.pkl'))
bnb_model = joblib.load(os.path.join(MODEL_DIR, 'bnb_model.pkl'))
xgb_model = joblib.load(os.path.join(MODEL_DIR, 'xgb_model.pkl'))

# vectorizers for text processing
vectorizers = joblib.load(os.path.join(MODEL_DIR, 'vectorizer.pkl'))

# label encoder for class mapping
le = joblib.load(os.path.join(MODEL_DIR, 'label_encoder.pkl'))

# unigram and bigram tfidf
tfidf_uni = vectorizers['unigram']
tfidf_bi = vectorizers['bigram']

# All class names in the order sklearn assigned them in alphabetical
CLASSES = list(le.classes_)

print("Models loaded")
print("\nLabel mapping")
for idx, label in enumerate(le.classes_):
    print(f"{idx} to {label}")

Models loaded

Label mapping
0 to Anxiety
1 to Depression
2 to Normal
3 to Stress
4 to Suicidal


# Model Weights

In [7]:
# One weight per model applied
# Based on overall training accuracy LR with highest weight , XGB boost 2nd while BNB weakest
MODEL_WEIGHTS = {
    'lr' : 0.50,
    'bnb': 0.20,
    'xgb': 0.30,
}

print("Model weights")
for model, w in MODEL_WEIGHTS.items():
    print(f"{model.upper()} gets {w}")

Model weights
LR gets 0.5
BNB gets 0.2
XGB gets 0.3


# Combined Model Voting Prediction

In [8]:
def preprocess(text: str):
    clean = lemmatize_text(remove_patterns(fix_encoding(text)))
    row = pd.DataFrame([{'text': text}])
    uni = tfidf_uni.transform([clean])
    bi = tfidf_bi.transform([clean])
    num = csr_matrix(build_features(row))
    return hstack([uni, bi, num])


# Runs all 3 models and combines their results using weights and returns individual predictions and final result
def predict_label(text: str) -> dict:

    X = preprocess(text)

    # Logistic Regression gives probabilities for all classes
    lr_proba  = lr_model.predict_proba(X)[0]

    # XGBoost also gives probabilities
    xgb_proba = xgb_model.predict_proba(X)[0]

    # But Naive Bayes only gives one class, so convert to one hot for example if sucidal it's going to be [0, 0, 0, 0, 1]
    bnb_raw  = bnb_model.predict(X)[0]
    bnb_vote = np.zeros(len(CLASSES))
    bnb_vote[bnb_raw] = 1.0

    # Combine predictions from all models using weighted voting
    combined = (MODEL_WEIGHTS['lr']  * lr_proba  + MODEL_WEIGHTS['bnb'] * bnb_vote  + MODEL_WEIGHTS['xgb'] * xgb_proba)

    # Get the final prediction where highest score wins
    final_idx   = np.argmax(combined)
    final_label = le.inverse_transform([final_idx])[0]

    return {
        'lr_pred'    : le.inverse_transform([np.argmax(lr_proba)])[0],
        'bnb_pred'   : le.inverse_transform([bnb_raw])[0],
        'xgb_pred'   : le.inverse_transform([np.argmax(xgb_proba)])[0],
        'final_label': final_label,
        'scores'     : dict(zip(CLASSES, combined.round(4)))
    }

# Crisis Keyword Routing System

In [9]:
# Crisis keywords that override the model for safety
KEYWORD_CRISIS = {
    'kill myself', 'end my life', 'want to die', 'going to kill',
    'suicide', 'overdose', 'goodbye forever', 'end it all',
    'no reason to live', 'better off dead', 'cant go on',
    'take my own life', 'dont want to be here', 'ready to die',
    'life isnt worth', 'nothing left to live for', 'final goodbye',
    'jumping off', 'hanging myself', 'cut my wrists', 'slit my',
    'self harm', 'hurt myself badly', 'harm myself',
    'want it to end', 'ready to go', 'cant take it anymore',
    'everyone better without me', 'burden to everyone',
    'world without me', 'stockpiling pills', 'writing goodbye',
    'suicide note', 'plan to end', 'wont be here tomorrow',
    'final message', 'kill me', 'wish i was dead',
    'shouldnt exist', 'disappear forever', 'cease to exist'
}


def route_message(user_text: str) -> dict:
    lower = user_text.lower()

    # keyword check if match found then instantly route to crisis by skipping ML model
    found_keyword = False

    for kw in KEYWORD_CRISIS:
        if kw in lower:
            found_keyword = True
            break

    if found_keyword == True:
        return {
            'message': user_text,
            'crisis_or_not': 'crisis',
            'label_predicted': 'Suicidal',
            'route': 'crisis',
            'triggered_by': 'keyword',
            'model_detail': None
        }

    # ML model prediction only if no keyword match
    detail = predict_label(user_text)
    label = detail['final_label']

    # Decide final route based on model prediction
    if label == 'Suicidal':
        route_status = 'crisis'
        crisis_flag = 'crisis'
    else:
        route_status = 'non_crisis'
        crisis_flag = 'not_crisis'

    return {
        'message': user_text,
        'crisis_or_not': crisis_flag,
        'label_predicted': label,
        'route': route_status,
        'triggered_by': 'model',
        'model_detail': detail
    }

# Testing

In [10]:
user_message = "Today was a really hot day i wish i ate an ice cream"
result = route_message(user_message)

# Only ouput message , crisis or not and label predicted
output = {
    'message': result['message'],
    'crisis_or_not': result['crisis_or_not'],
    'label_predicted': result['label_predicted'],
    }

print(output)

{'message': 'Today was a really hot day i wish i ate an ice cream', 'crisis_or_not': 'not_crisis', 'label_predicted': 'Normal'}


# Gemini API Setup and Configuration

In [27]:
# Load environment variables
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

genai.configure(api_key=GEMINI_API_KEY)

# Initialize Chat Memory and User Profile

In [28]:
conversation_history = []

user_profile = {
    'summary': 'New user with no conversation history yet.',
    'dominant_emotions': [],
    'crisis_count': 0,
    'total_messages': 0
}

print(f"User profile: {user_profile}")

User profile: {'summary': 'New user with no conversation history yet.', 'dominant_emotions': [], 'crisis_count': 0, 'total_messages': 0}


# Setup Folders for Data and Database

In [29]:
RAG_DOCS_DIR = '../data/rag_documents'
CHROMA_DB_DIR = '../data/chroma_db'
os.makedirs(RAG_DOCS_DIR, exist_ok=True)
os.makedirs(CHROMA_DB_DIR, exist_ok=True)

# Initialize AI Models and Settings

In [32]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",   # Converts text into numbers
    google_api_key=GEMINI_API_KEY
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=GEMINI_API_KEY,
    temperature=0.7,
    max_output_tokens=1024
)

response = llm.invoke("Hello how are you")

print(response.text)

Hello! I'm doing well, thank you for asking. How are you doing today? Is there anything I can help you with?
